# Basic operations

In [26]:
import polars as pl
import numpy as np

In [27]:
np.random.seed(42)

df = pl.DataFrame(
    {
        "nrs": [1, 2, 3, None, 5],
        "names": ["foo", "ham", "spam", "egg", "spam"],
        "random": np.random.rand(5),
        "groups": ["A", "A", "B", "A", "B"],
    }
)

df

nrs,names,random,groups
i64,str,f64,str
1,"""foo""",0.37454,"""A"""
2,"""ham""",0.950714,"""A"""
3,"""spam""",0.731994,"""B"""
null,"""egg""",0.598658,"""A"""
5,"""spam""",0.156019,"""B"""


In [28]:
result = df.with_columns(
    (pl.col("nrs") + 5).alias("nrs + 5"),
    (pl.col("nrs") - 5).alias("nrs - 5"),
    (pl.col("nrs") * pl.col("random")).alias("nrs * random"),
    (pl.col("nrs") / pl.col("random")).alias("nrs / random"),
    (pl.col("nrs") ** 2).alias("nrs ** 2"),
    (pl.col("nrs") % 3).alias("nrs % 3"),
)

result

nrs,names,random,groups,nrs + 5,nrs - 5,nrs * random,nrs / random,nrs ** 2,nrs % 3
i64,str,f64,str,i64,i64,f64,f64,i64,i64
1,"""foo""",0.37454,"""A""",6,-4,0.37454,2.669941,1,1
2,"""ham""",0.950714,"""A""",7,-3,1.901429,2.103681,4,2
3,"""spam""",0.731994,"""B""",8,-2,2.195982,4.098395,9,0
null,"""egg""",0.598658,"""A""",null,null,null,null,null,null
5,"""spam""",0.156019,"""B""",10,0,0.780093,32.047453,25,2


In [29]:
result_named_operators = df.with_columns(
    (pl.col("nrs").add(5)).alias("nrs + 5"),
    (pl.col("nrs").sub(5)).alias("nrs - 5"),
    (pl.col("nrs").mul(pl.col("random"))).alias("nrs * random"),
    (pl.col("nrs").truediv(pl.col("random"))).alias("nrs / random"),
    (pl.col("nrs").pow(2)).alias("nrs ** 2"),
    (pl.col("nrs").mod(3)).alias("nrs % 3"),
)

result.equals(result_named_operators)

True

In [30]:
result = df.select(
    (pl.col("nrs") > 1).alias("nrs > 1"),
    (pl.col("nrs") >= 3).alias("nrs >= 3"),
    (pl.col("random") < 0.2).alias("random < .2"),
    (pl.col("random") <= 0.5).alias("random <= .5"),
    (pl.col("nrs") != 1).alias("nrs != 1"),
    (pl.col("nrs") == 1).alias("nrs == 1"),
)

result

nrs > 1,nrs >= 3,random < .2,random <= .5,nrs != 1,nrs == 1
bool,bool,bool,bool,bool,bool
false,false,false,true,false,true
true,false,false,false,true,false
true,true,false,false,true,false
null,null,false,false,null,null
true,true,true,true,true,false


In [31]:
result = df.select(
    (pl.col("nrs").gt(1)).alias("nrs > 1"),
    (pl.col("nrs").ge(3)).alias("nrs >= 3"),
    (pl.col("random").lt(0.2)).alias("random < .2"),
    (pl.col("random").le(0.5)).alias("random <= .5"),
    (pl.col("nrs").ne(1)).alias("nrs != 1"),
    (pl.col("nrs").eq(1)).alias("nrs == 1"),
)

result

nrs > 1,nrs >= 3,random < .2,random <= .5,nrs != 1,nrs == 1
bool,bool,bool,bool,bool,bool
false,false,false,true,false,true
true,false,false,false,true,false
true,true,false,false,true,false
null,null,false,false,null,null
true,true,true,true,true,false


## Boolean and bitwise operations

In [32]:
# Boolean operators & | ~
result = df.select(
    ((~pl.col("nrs").is_null()) & (pl.col("groups") == "A")).alias(
        "number not null and group A"
    ),
    ((pl.col("random") < 0.5) | (pl.col("groups") == "B")).alias(
        "random < 0.5 or group B"
    ),
)

result

number not null and group A,random < 0.5 or group B
bool,bool
true,true
true,false
false,true
false,false
false,true


In [33]:
result2 = df.select(
    (pl.col("nrs").is_null().not_().and_(pl.col("groups") == "A")).alias(
        "number not null and group A"
    ),
    ((pl.col("random") < 0.5).or_(pl.col("groups") == "B")).alias(
        "random < 0.5 or group B"
    ),
)

result.equals(result2)

True

## Counting (unique) values

In [35]:
long_df = pl.DataFrame({"numbers": np.random.randint(0, 100_000, 100_000)})

result = long_df.select(
    pl.col("numbers").n_unique().alias("n_unique"),
    pl.col("numbers").approx_n_unique().alias("approx_n_unique"),
)

result

n_unique,approx_n_unique
u32,u32
63217,63699


In [36]:
result = df.select(
    pl.col("names").value_counts().alias("value_counts"),
)

result

value_counts
struct[2]
"{""egg"",1}"
"{""spam"",2}"
"{""ham"",1}"
"{""foo"",1}"


In [56]:
result = (
    df
    .select(pl.col("names").value_counts().alias("value_counts"))
    .unnest("value_counts")
    .sort("count")
)

result

names,count
str,u32
"""foo""",1
"""egg""",1
"""ham""",1
"""spam""",2


In [59]:
result = (
    df
    .select(
        pl.col("names").unique().alias("unique"),
        pl.col("names").unique_counts().alias("unique_counts"),
    )
    .sort("unique_counts")
)

result

unique,unique_counts
str,u32
"""ham""",1
"""spam""",1
"""egg""",1
"""foo""",2


## Conditionals (ternary operations)

In [ ]:
result = df.select(
    pl.col("nrs"),
    pl.when(pl.col("nrs") % 2 == 1)
        .then(3 * pl.col("nrs") + 1)
        .otherwise(pl.col("nrs") // 2)
        .alias("collatz"),
)

result

nrs,collatz
i64,i64
1,4
2,1
3,10
null,null
5,16
